In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [8]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_mensajeria = config['MENSAJERIA_OLTP']
    config_etl = config['ETL_PROCESS']

# Construct the database URL
url_mensajeria = (f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
          f"{config_mensajeria['port']}/{config_mensajeria['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

In [21]:
dim_sede = pd.read_sql_table('sede',mensajeria)
dim_sede.head()


,sede_id,nombre,direccion,telefono,nombre_contacto,ciudad_id,cliente_id
0,10,FARALLONES /123,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
1,11,REMEDIOZ/ 123,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
2,13,DIME / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
3,14,DESPACHOS / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
4,23,POPAYAN BODEGA 28 / A,Los angeles distrito Latino,310-70000,JUAN PEREZ,3,11


In [22]:
dim_sede.drop(columns=['nombre_contacto','direccion','telefono','ciudad_id'],inplace=True)

In [23]:
dim_sede[dim_sede.duplicated(subset='nombre')]

,sede_id,nombre,cliente_id
39,47,CLINICA CALI,7
46,53,NUEVA HEMATO,8
47,54,NUEVA HEMATO,8


In [24]:
dim_sede.drop_duplicates(inplace=True)

In [25]:
dim_sede = dim_sede.reset_index()  # sin drop=True
dim_sede = dim_sede.rename(columns={"index": "key_dim_sede"})  # solo si hiciera falta


fila_no_aplica = pd.DataFrame(
    [
        {
            "key_dim_sede": -1,
            "sede_id": -1,
            "nombre": "No aplica - dirección puntual",
            "cliente_id": -1,
        }
    ]
)

dim_sede = pd.concat([dim_sede, fila_no_aplica], ignore_index=True)

In [26]:
dim_sede = dim_sede.set_index("key_dim_sede")

In [27]:
dim_sede.to_sql('dim_sede', etl_conn, if_exists='replace',index_label='key_dim_sede')

53